In [1]:
import pandas as pd
import numpy as np

matches = pd.read_csv(
    "../../data/processed/match_data/matches_with_form.csv"
)

matches["date"] = pd.to_datetime(matches["date"])

matches = matches.sort_values("date").reset_index(drop=True)

goals_scored_history = {}
goals_conceded_history = {}
home_avg_scored = []
away_avg_scored = []

home_avg_conceded = []
away_avg_conceded = []

for _, row in matches.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    home_scored_hist = goals_scored_history.get(home, [])
    away_scored_hist = goals_scored_history.get(away, [])

    home_conceded_hist = goals_conceded_history.get(home, [])
    away_conceded_hist = goals_conceded_history.get(away, [])

    home_avg_scored.append(
        np.mean(home_scored_hist[-5:])
        if len(home_scored_hist) > 0
        else 0
    )

    away_avg_scored.append(
        np.mean(away_scored_hist[-5:])
        if len(away_scored_hist) > 0
        else 0
    )

    home_avg_conceded.append(
        np.mean(home_conceded_hist[-5:])
        if len(home_conceded_hist) > 0
        else 0
    )

    away_avg_conceded.append(
        np.mean(away_conceded_hist[-5:])
        if len(away_conceded_hist) > 0
        else 0
    )
    home_goals = row["home_score"]
    away_goals = row["away_score"]

    goals_scored_history.setdefault(
        home, []
    ).append(home_goals)

    goals_conceded_history.setdefault(
        home, []
    ).append(away_goals)

    goals_scored_history.setdefault(
        away, []
    ).append(away_goals)

    goals_conceded_history.setdefault(
        away, []
    ).append(home_goals)


matches["home_avg_goals_scored_5"] = home_avg_scored
matches["away_avg_goals_scored_5"] = away_avg_scored

matches["home_avg_goals_conceded_5"] = home_avg_conceded
matches["away_avg_goals_conceded_5"] = away_avg_conceded

matches["goals_scored_diff"] = (
    matches["home_avg_goals_scored_5"]
    - matches["away_avg_goals_scored_5"]
)

matches["goals_conceded_diff"] = (
    matches["home_avg_goals_conceded_5"]
    - matches["away_avg_goals_conceded_5"]
)

print(
    matches[
        [
            "home_avg_goals_scored_5",
            "away_avg_goals_scored_5",
            "home_avg_goals_conceded_5",
            "away_avg_goals_conceded_5"
        ]
    ].describe()
)
matches.to_csv(
    "../../data/processed/training_data/feature_v1_dataset.csv",
    index=False
)


       home_avg_goals_scored_5  away_avg_goals_scored_5  \
count             12612.000000             12612.000000   
mean                  1.308171                 1.268822   
std                   0.769064                 0.742522   
min                   0.000000                 0.000000   
25%                   0.800000                 0.800000   
50%                   1.200000                 1.200000   
75%                   1.800000                 1.600000   
max                   8.000000                 7.000000   

       home_avg_goals_conceded_5  away_avg_goals_conceded_5  
count               12612.000000               12612.000000  
mean                    1.268501                   1.303802  
std                     0.787585                   0.800663  
min                     0.000000                   0.000000  
25%                     0.800000                   0.800000  
50%                     1.200000                   1.200000  
75%                     1.600000  